# Lab 3

In [26]:
# !pip install -q \
#     "transformers==4.46.3" \
#     "datasets==3.1.0" \
#     "evaluate==0.4.3" \
#     "jiwer==3.0.5" \
#     "pytorch-lightning==2.4.0" \
#     "librosa==0.10.2.post1" \
#     "soundfile==0.12.1" \
#     "sentencepiece==0.2.0" \
#     "accelerate==1.0.1"

In [27]:
# import torch
# import transformers
# import datasets
# import evaluate
# import pytorch_lightning as pl
# import torchaudio
# import pandas as pd

# print("torch", torch.__version__)
# print("transformers", transformers.__version__)
# print("datasets", datasets.__version__)
# print("evaluate", evaluate.__version__)
# print("pl", pl.__version__)
# print("torchaudio", torchaudio.__version__)
# print("pandas", pd.__version__)

In [28]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
2
Tesla T4


In [29]:
import os
import gc
import re
import json
import math
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torchaudio
import pytorch_lightning as pl
import evaluate

from torch.utils.data import Dataset, DataLoader
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    get_linear_schedule_with_warmup,
)
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
pl.seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision("medium")

INFO:lightning_fabric.utilities.seed:Seed set to 42


# 1. Config


In [30]:
TEST_LINES = {
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
}

def detect_dataset_root():
    candidates = [
        Path("./dataset"),
        Path("/kaggle/input/dataset/dataset"),
        Path("/kaggle/input/dataset"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.iterdir():
            if sub.is_dir():
                candidates.append(sub / "dataset")
                candidates.append(sub)

    for p in candidates:
        if p.exists():
            return str(p)
    return "./dataset"

IS_KAGGLE = Path("/kaggle").exists()

CFG = {
    "dataset_root": detect_dataset_root(),
    "labels_path": None,
    "model_name": "openai/whisper-small",
    "language": "ukrainian",
    "task": "transcribe",
    "sample_rate": 16000,
    "max_audio_seconds": 30.0,
    "train_batch_size": 4,
    "eval_batch_size": 4,
    "grad_accum": 4,
    "num_workers": 2,
    "learning_rate": 1e-5,
    "weight_decay": 1e-2,
    "warmup_ratio": 0.05,
    "max_epochs": 5,
    "val_ratio": 0.1,
    "limit_train_samples": None,
    "limit_val_samples": None,
    "limit_test_samples": None,
    "freeze_encoder": False,
    "fp16": True,
    "output_dir": "/kaggle/working/whisper_toronto_lab3" if IS_KAGGLE else "./whisper_toronto_lab3",
}

os.makedirs(CFG["output_dir"], exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

print("IS_KAGGLE:", IS_KAGGLE)
print("device:", device)
print("dataset_root:", CFG["dataset_root"])
print("output_dir:", CFG["output_dir"])

IS_KAGGLE: True
device: cuda
dataset_root: /kaggle/input/datasets
output_dir: /kaggle/working/whisper_toronto_lab3


## 2. Helper functions

In [31]:
def normalize_text(text: str) -> str:
    text = str(text).strip()
    text = text.replace("’", "'").replace("`", "'").replace("ʼ", "'")
    text = re.sub(r"\s+", " ", text)
    return text

def find_labels_file(dataset_root: str):
    root = Path(dataset_root)
    candidates = [
        root / "labels.json",
        root / "labels.jsonl",
        root / "dataset_example" / "labels.json",
        root / "dataset_example" / "labels.jsonl",
    ]
    for p in candidates:
        if p.exists():
            return p
    extra = list(root.rglob("labels.json")) + list(root.rglob("labels.jsonl"))
    if extra:
        return extra[0]
    raise FileNotFoundError("Не знайшов labels.json або labels.jsonl.")

def _extract_path_and_text(obj):
    path_keys = ["path", "audio_path", "audio", "file", "wav", "audio_filepath"]
    text_keys = ["text", "sentence", "transcript", "transcription", "label", "normalized_text"]

    path_val, text_val = None, None

    if isinstance(obj, dict):
        for k in path_keys:
            if isinstance(obj.get(k), str):
                path_val = obj[k]
                break

        for k in text_keys:
            if isinstance(obj.get(k), str):
                text_val = obj[k]
                break

        if path_val is None:
            for k, v in obj.items():
                if isinstance(v, str) and v.lower().endswith((".wav", ".mp3", ".flac", ".m4a")):
                    path_val = v
                    break

        if path_val is None and text_val is None and len(obj) == 1:
            k, v = next(iter(obj.items()))
            if isinstance(k, str) and isinstance(v, str):
                path_val, text_val = k, v

    return path_val, text_val

def load_labels(labels_path):
    labels_path = Path(labels_path)

    if labels_path.suffix == ".json":
        with open(labels_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            return {
                str(k): normalize_text(v)
                for k, v in data.items()
                if isinstance(k, str) and isinstance(v, str)
            }

        if isinstance(data, list):
            parsed = {}
            for obj in data:
                path_val, text_val = _extract_path_and_text(obj)
                if path_val is not None and text_val is not None:
                    parsed[path_val] = normalize_text(text_val)
            return parsed

        raise ValueError("Непідтримуваний формат labels.json")

    if labels_path.suffix == ".jsonl":
        parsed = {}
        with open(labels_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                obj = json.loads(line)

                # ВАЖЛИВО: якщо рядок = великий dict path -> text
                if isinstance(obj, dict):
                    all_str_pairs = all(isinstance(k, str) and isinstance(v, str) for k, v in obj.items())
                    if all_str_pairs:
                        for k, v in obj.items():
                            parsed[k] = normalize_text(v)
                        continue

                # Класичний випадок: один dict із полями path/text
                path_val, text_val = _extract_path_and_text(obj)
                if path_val is not None and text_val is not None:
                    parsed[path_val] = normalize_text(text_val)

        return parsed

    raise ValueError(f"Unsupported labels format: {labels_path}")

In [32]:
if CFG["labels_path"] is None:
    CFG["labels_path"] = str(find_labels_file(CFG["dataset_root"]))

labels_map = load_labels(CFG["labels_path"])

print("labels_path:", CFG["labels_path"])
print("num labels:", len(labels_map))
print("sample item:", next(iter(labels_map.items())))

labels_path: /kaggle/input/datasets/kravchukandrii/dataset/labels.jsonl
num labels: 29232
sample item: ('dataset/toronto_157/toronto_157_0.wav', 'Слава Ісу! Ви сі дивите програму «Грати, песик, дужка, гривня, знак питання, долар, нуль» ₴?$0»). Я є її ведучий, Майкл Щур. Вйо до новин!')


## 3. Dataframe

In [ ]:
def resolve_audio_path(key: str, dataset_root: str, labels_path: str, wav_index: dict):
    key = str(key).strip().replace("\\", "/")
    key_path = Path(key)

    candidates = []

    candidates.append(key_path)

    candidates.append(Path(dataset_root) / key_path)

    parts = key_path.parts
    if len(parts) > 0 and parts[0] == "dataset":
        stripped = Path(*parts[1:])
        candidates.append(Path(dataset_root) / stripped)
        candidates.append(Path(labels_path).parent / stripped)

    candidates.append(Path(labels_path).parent / key_path)

    for c in candidates:
        if c.exists():
            return c.resolve()

    return wav_index.get(key_path.name, None)


dataset_root_path = Path(CFG["dataset_root"])

all_wavs = list(dataset_root_path.rglob("*.wav"))
wav_index = {p.name: p.resolve() for p in all_wavs}

print("indexed wav files:", len(wav_index))

records = []
missing_paths = []

for rel_path, text in labels_map.items():
    audio_path = resolve_audio_path(rel_path, CFG["dataset_root"], CFG["labels_path"], wav_index)

    if audio_path is None:
        missing_paths.append(rel_path)
        continue

    line_id = None
    for part in audio_path.parts:
        if part.startswith("toronto_") and part.count("_") == 1:
            line_id = part
            break

    if line_id is None and audio_path.parent.name.startswith("toronto_"):
        line_id = audio_path.parent.name

    records.append(
        {
            "audio_path": str(audio_path),
            "rel_path": rel_path,
            "text": normalize_text(text),
            "line_id": line_id,
            "utt_id": audio_path.stem,
        }
    )

df = pd.DataFrame(records).drop_duplicates(subset=["audio_path"]).reset_index(drop=True)
df = df[df["text"].str.len() > 0].copy()

print("resolved samples:", len(df))
print("missing paths:", len(missing_paths))
if missing_paths:
    print("first missing:", missing_paths[:5])

df.head()

indexed wav files: 18303
resolved samples: 18199
missing paths: 10929
first missing: ['dataset/toronto_155/toronto_155_43.wav', 'dataset/toronto_155/toronto_155_44.wav', 'dataset/toronto_155/toronto_155_45.wav', 'dataset/toronto_155/toronto_155_46.wav', 'dataset/toronto_155/toronto_155_47.wav']


,audio_path,rel_path,text,line_id,utt_id
0,/kaggle/input/datasets/kravchukandrii/dataset/...,dataset/toronto_157/toronto_157_0.wav,"Слава Ісу! Ви сі дивите програму «Грати, песик...",toronto_157,toronto_157_0
1,/kaggle/input/datasets/kravchukandrii/dataset/...,dataset/toronto_157/toronto_157_1.wav,"Купол Верховної Ради впав під час засідання, н...",toronto_157,toronto_157_1
2,/kaggle/input/datasets/kravchukandrii/dataset/...,dataset/toronto_157/toronto_157_2.wav,"живих добивають з автоматів, кров, вогонь, аго...",toronto_157,toronto_157_2
3,/kaggle/input/datasets/kravchukandrii/dataset/...,dataset/toronto_157/toronto_157_3.wav,саме такий теракт вона планувала влаштувати у ...,toronto_157,toronto_157_3
4,/kaggle/input/datasets/kravchukandrii/dataset/...,dataset/toronto_157/toronto_157_4.wav,"що Надія Савченко особисто планувала,",toronto_157,toronto_157_4


In [36]:
import soundfile as sf

def get_audio_duration_sec(path: str):
    info = sf.info(path)
    return float(info.frames) / float(info.samplerate)

df["duration_sec"] = df["audio_path"].apply(get_audio_duration_sec)
df = df[df["duration_sec"] <= CFG["max_audio_seconds"]].reset_index(drop=True)

print(df["duration_sec"].describe())
print("after duration filter:", len(df))

count    18199.000000
mean         5.956902
std          1.235828
min          0.140000
25%          5.310000
50%          6.200000
75%          6.880000
max         16.020000
Name: duration_sec, dtype: float64
after duration filter: 18199


## 4. Train / Val / Test split without data leakage

In [38]:
test_df = df[df["line_id"].isin(TEST_LINES)].copy().reset_index(drop=True)
trainval_df = df[~df["line_id"].isin(TEST_LINES)].copy().reset_index(drop=True)

if len(test_df) == 0:
    raise ValueError("Test split порожній. Перевір labels і структуру dataset.")

unique_lines = trainval_df["line_id"].dropna().unique().tolist()
random.Random(SEED).shuffle(unique_lines)

n_val_lines = max(1, int(len(unique_lines) * CFG["val_ratio"]))
val_lines = set(unique_lines[:n_val_lines])

val_df = trainval_df[trainval_df["line_id"].isin(val_lines)].copy().reset_index(drop=True)
train_df = trainval_df[~trainval_df["line_id"].isin(val_lines)].copy().reset_index(drop=True)

if CFG["limit_train_samples"] is not None:
    train_df = train_df.iloc[:CFG["limit_train_samples"]].copy()
if CFG["limit_val_samples"] is not None:
    val_df = val_df.iloc[:CFG["limit_val_samples"]].copy()
if CFG["limit_test_samples"] is not None:
    test_df = test_df.iloc[:CFG["limit_test_samples"]].copy()

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

assert len(set(train_df["line_id"]) & set(val_df["line_id"])) == 0
assert len(set(train_df["line_id"]) & set(test_df["line_id"])) == 0
assert len(set(val_df["line_id"]) & set(test_df["line_id"])) == 0

train: 11424
val: 1258
test: 5517


In [39]:
display(
    pd.DataFrame(
        {
            "split": ["train", "val", "test"],
            "samples": [len(train_df), len(val_df), len(test_df)],
            "hours": [
                train_df["duration_sec"].sum() / 3600,
                val_df["duration_sec"].sum() / 3600,
                test_df["duration_sec"].sum() / 3600,
            ],
            "unique_lines": [
                train_df["line_id"].nunique(),
                val_df["line_id"].nunique(),
                test_df["line_id"].nunique(),
            ],
        }
    )
)

,split,samples,hours,unique_lines
0,train,11424,18.911943,45
1,val,1258,2.073241,5
2,test,5517,9.128611,21


## 5. Whisper processor + model

In [41]:
processor = WhisperProcessor.from_pretrained(
    CFG["model_name"],
    language=CFG["language"],
    task=CFG["task"],
)
model = WhisperForConditionalGeneration.from_pretrained(CFG["model_name"])

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=CFG["language"],
    task=CFG["task"],
)
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.language = CFG["language"]
model.generation_config.task = CFG["task"]

if CFG["freeze_encoder"]:
    model.model.encoder.requires_grad_(False)

print("Model loaded:", CFG["model_name"])

Model loaded: openai/whisper-small


## 6. Dataset and collator

In [42]:
class TorontoWhisperDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, processor: WhisperProcessor, sample_rate: int = 16000):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio, sr = torchaudio.load(row["audio_path"])
        audio = audio.mean(dim=0)

        if sr != self.sample_rate:
            audio = torchaudio.functional.resample(audio, sr, self.sample_rate)

        audio = audio.numpy()

        input_features = self.processor.feature_extractor(
            audio,
            sampling_rate=self.sample_rate,
            return_tensors="pt",
        ).input_features[0]

        labels = self.processor.tokenizer(
            row["text"],
            return_tensors="pt",
            truncation=True,
        ).input_ids[0]

        return {
            "input_features": input_features,
            "labels": labels,
            "text": row["text"],
            "audio_path": row["audio_path"],
            "utt_id": row["utt_id"],
            "line_id": row["line_id"],
        }

@dataclass
class WhisperCollator:
    processor: WhisperProcessor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        if labels.size(1) > 0 and (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        batch["texts"] = [f["text"] for f in features]
        batch["audio_paths"] = [f["audio_path"] for f in features]
        batch["utt_ids"] = [f["utt_id"] for f in features]
        batch["line_ids"] = [f["line_id"] for f in features]
        return batch

In [43]:
train_ds = TorontoWhisperDataset(train_df, processor, CFG["sample_rate"])
val_ds = TorontoWhisperDataset(val_df, processor, CFG["sample_rate"])
test_ds = TorontoWhisperDataset(test_df, processor, CFG["sample_rate"])

collator = WhisperCollator(processor)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["train_batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collator,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["eval_batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collator,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["eval_batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collator,
)

batch = next(iter(train_loader))
{k: (v.shape if torch.is_tensor(v) else len(v)) for k, v in batch.items()}

{'input_features': torch.Size([4, 80, 3000]),
 'labels': torch.Size([4, 61]),
 'texts': 4,
 'audio_paths': 4,
 'utt_ids': 4,
 'line_ids': 4}

## 7. Metrics

In [44]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def postprocess_text(texts):
    return [normalize_text(t).strip() for t in texts]

def compute_metrics(pred_texts, ref_texts):
    pred_texts = postprocess_text(pred_texts)
    ref_texts = postprocess_text(ref_texts)
    return {
        "wer": wer_metric.compute(predictions=pred_texts, references=ref_texts),
        "cer": cer_metric.compute(predictions=pred_texts, references=ref_texts),
    }

## 8. Lightning module

In [45]:
class WhisperLightningModule(pl.LightningModule):
    def __init__(self, model, processor, train_steps: int, cfg: dict):
        super().__init__()
        self.model = model
        self.processor = processor
        self.train_steps = train_steps
        self.cfg = cfg
        self.save_hyperparameters(ignore=["model", "processor"])

    def training_step(self, batch, batch_idx):
        outputs = self.model(
            input_features=batch["input_features"],
            labels=batch["labels"],
        )
        loss = outputs.loss
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=batch["input_features"].size(0))
        return loss

    @torch.no_grad()
    def shared_eval_step(self, batch, stage="val"):
        outputs = self.model(
            input_features=batch["input_features"],
            labels=batch["labels"],
        )
        loss = outputs.loss

        generated_ids = self.model.generate(
            input_features=batch["input_features"],
            max_new_tokens=128,
        )

        label_ids = batch["labels"].clone()
        label_ids[label_ids == -100] = self.processor.tokenizer.pad_token_id

        pred_texts = self.processor.batch_decode(generated_ids, skip_special_tokens=True)
        ref_texts = self.processor.batch_decode(label_ids, skip_special_tokens=True)

        metrics = compute_metrics(pred_texts, ref_texts)

        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True, batch_size=batch["input_features"].size(0))
        self.log(f"{stage}_wer", metrics["wer"], prog_bar=True, on_epoch=True, batch_size=batch["input_features"].size(0))
        self.log(f"{stage}_cer", metrics["cer"], prog_bar=True, on_epoch=True, batch_size=batch["input_features"].size(0))
        return loss

    def validation_step(self, batch, batch_idx):
        return self.shared_eval_step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self.shared_eval_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.cfg["learning_rate"],
            weight_decay=self.cfg["weight_decay"],
        )
        warmup_steps = int(self.train_steps * self.cfg["warmup_ratio"])
        scheduler = get_linear_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=self.train_steps,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
            },
        }

## 9. Trainer

In [47]:
steps_per_epoch = max(1, math.ceil(len(train_loader) / CFG["grad_accum"]))
train_steps = steps_per_epoch * CFG["max_epochs"]

lit_model = WhisperLightningModule(model, processor, train_steps=train_steps, cfg=CFG)

checkpoint_cb = ModelCheckpoint(
    dirpath=os.path.join(CFG["output_dir"], "checkpoints"),
    filename="whisper-{epoch:02d}-{val_wer:.4f}",
    monitor="val_wer",
    mode="min",
    save_top_k=1,
    save_last=True,
)

early_stop_cb = EarlyStopping(monitor="val_wer", mode="min", patience=2)
lr_monitor = LearningRateMonitor(logging_interval="step")
logger = CSVLogger(save_dir=CFG["output_dir"], name="logs")

trainer = pl.Trainer(
    default_root_dir=CFG["output_dir"],
    max_epochs=CFG["max_epochs"],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() and CFG["fp16"] else 32,
    accumulate_grad_batches=CFG["grad_accum"],
    gradient_clip_val=1.0,
    logger=logger,
    callbacks=[checkpoint_cb, early_stop_cb, lr_monitor],
    log_every_n_steps=5,
)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


## 10. Training

In [48]:
trainer.fit(lit_model, train_loader, val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type                            | Params | Mode
-----------------------------------------------------------------
0 | model | WhisperForConditionalGeneration | 241 M  | eval
-----------------------------------------------------------------
240 M     Trainable params
1.2 M     Non-trainable params
241 M     Total params
966.940   Total estimated model params size (MB)
0         Modules in train mode
350       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
You have passed task=transcribe, but also have set `forced_decoder_ids` to [(1, 50280), (2, 50359), (3, 50363)] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Training: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/module.py:1271: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


## 11. Best checkpoint

In [49]:
best_ckpt = checkpoint_cb.best_model_path
print("best checkpoint:", best_ckpt)

best_model = WhisperLightningModule.load_from_checkpoint(
    best_ckpt,
    model=WhisperForConditionalGeneration.from_pretrained(CFG["model_name"]),
    processor=processor,
    train_steps=train_steps,
    cfg=CFG,
)

best_model.model.generation_config.forced_decoder_ids = forced_decoder_ids
best_model.model.generation_config.language = CFG["language"]
best_model.model.generation_config.task = CFG["task"]
best_model.eval().to(device)

best checkpoint: /kaggle/working/whisper_toronto_lab3/checkpoints/whisper-epoch=02-val_wer=0.3150.ckpt


WhisperLightningModule(
  (model): WhisperForConditionalGeneration(
    (model): WhisperModel(
      (encoder): WhisperEncoder(
        (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
        (embed_positions): Embedding(1500, 768)
        (layers): ModuleList(
          (0-11): 12 x WhisperEncoderLayer(
            (self_attn): WhisperSdpaAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=False)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            

## 12. Testing on holdout test set

In [50]:
trainer.test(best_model, dataloaders=test_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_cer          │    0.15155573189258575    │
│         test_loss         │    0.5186328291893005     │
│         test_wer          │    0.3698369562625885     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.5186328291893005,
  'test_wer': 0.3698369562625885,
  'test_cer': 0.15155573189258575}]

## 13. Detailed inference on test and predictions

In [53]:
@torch.no_grad()
def predict_dataset(model_module, dataloader, processor, device="cpu"):
    model_module.eval()
    model_module.to(device)
    rows = []

    for batch in dataloader:
        input_features = batch["input_features"].to(device, non_blocking=True)

        generated_ids = model_module.model.generate(
            input_features=input_features,
            max_new_tokens=96,
            num_beams=1,
        )

        pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

        label_ids = batch["labels"].clone()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        ref_texts = processor.batch_decode(label_ids, skip_special_tokens=True)

        rows.extend(
            {
                "utt_id": utt_id,
                "line_id": line_id,
                "audio_path": audio_path,
                "reference": normalize_text(ref),
                "prediction": normalize_text(pred),
            }
            for utt_id, line_id, audio_path, pred, ref in zip(
                batch["utt_ids"], batch["line_ids"], batch["audio_paths"], pred_texts, ref_texts
            )
        )

    result_df = pd.DataFrame(rows)
    metrics = compute_metrics(
        result_df["prediction"].tolist(),
        result_df["reference"].tolist()
    )
    return result_df, metrics

pred_df, test_metrics = predict_dataset(best_model, test_loader, processor, device=device)
print(test_metrics)
pred_df.head(20)

{'wer': 0.3627294033272287, 'cer': 0.1434393517007632}


,utt_id,line_id,audio_path,reference,prediction
0,toronto_157_0,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,"Слава Ісу! Ви сі дивите програму «Грати, песик...","Слава Ісу! Ви сі дивите програму «Грати, песик..."
1,toronto_157_1,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,"Купол Верховної Ради впав під час засідання, н...","Купол Верховної Ради впав під час засідання, н..."
2,toronto_157_2,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,"живих добивають з автоматів, кров, вогонь, аго...","жилих додувають з автоматів. Кров, Богоня, Гон..."
3,toronto_157_3,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,саме такий теракт вона планувала влаштувати у ...,Саме такий теракт вона повновала влаштувати у ...
4,toronto_157_4,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,"що Надія Савченко особисто планувала,",що надія Савченко особисто планувала
5,toronto_157_5,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,"особисто вербувала, особисто розпо... давала в...",особисто виробувала. Особисто розпредавала в к...
6,toronto_157_6,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,"про те, як провести терористичний акт тут, в ц...","проте, як повести терористичний акт тут, в цьо..."
7,toronto_157_7,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,знищивши бойовими гранатами дві ложі: Урядову ...,знищивши бойовими гранатами дві ложі урядову і...
8,toronto_157_8,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,Мінометами обрушивши купол Верховної Ради і ав...,"Мінометами, обрушивши купол Верховної Ради, і ..."
9,toronto_157_9,toronto_157,/kaggle/input/datasets/kravchukandrii/dataset/...,що у неї взагалі у голові?! Що?!! Зараз ми діз...,"Да що у неї, взагалі, в голові? Що? Зараз ми і..."


In [54]:
pred_path = os.path.join(CFG["output_dir"], "test_predictions.csv")
pred_df.to_csv(pred_path, index=False, encoding="utf-8")
print("saved:", pred_path)

saved: /kaggle/working/whisper_toronto_lab3/test_predictions.csv


## 14. Saving final model

In [55]:
save_dir = os.path.join(CFG["output_dir"], "best_hf_model")
os.makedirs(save_dir, exist_ok=True)

best_model.model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)

print("saved model to:", save_dir)

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


saved model to: /kaggle/working/whisper_toronto_lab3/best_hf_model


## 15. Summary

In [56]:
summary = pd.DataFrame(
    [
        {
            "model_name": CFG["model_name"],
            "language": CFG["language"],
            "task": CFG["task"],
            "train_samples": len(train_df),
            "val_samples": len(val_df),
            "test_samples": len(test_df),
            "test_wer": test_metrics["wer"],
            "test_cer": test_metrics["cer"],
        }
    ]
)
summary

,model_name,language,task,train_samples,val_samples,test_samples,test_wer,test_cer
0,openai/whisper-small,ukrainian,transcribe,11424,1258,5517,0.362729,0.143439
